<a href="https://colab.research.google.com/github/sulucay01/multimodal-rs-segmentation/blob/dev/03_baseline_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Baselines

DI725 Term Project — Multimodal Fusion for Remote Sensing Land Cover Segmentation.

This notebook trains and evaluates the vision-only baselines used as reference points for the multimodal experiments in `04_multimodal.ipynb`:

1. UNetFormer (no augmentation) — reproduces the PoC baseline.
2. UNetFormer (with augmentation) — Phase 2 baseline.
3. SegFormer (with augmentation) — second Phase 2 baseline.

All models share the same train/val/test split (from `02_preprocessing.ipynb`), pre-converted class-index masks, weighted cross-entropy loss with median frequency balancing, and 30-epoch training schedule.

Sections:
1. Setup
2. Dataset and DataLoaders
3. UNetFormer architecture
4. UNetFormer baseline (no augmentation)
5. UNetFormer baseline (with augmentation)
6. SegFormer baseline (with augmentation)
7. Test set evaluation
8. Save results

## 1. Setup

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install dependencies not present in Colab by default
!pip install transformers timm einops -q

In [3]:
# Project paths (Drive) and a local working copy for fast I/O during training
from pathlib import Path
import shutil, os

PROJECT_DIR     = Path('/content/drive/MyDrive/DI725_Project')
DATA_DIR_DRIVE  = PROJECT_DIR / 'data'
SPLITS_CSV      = DATA_DIR_DRIVE / 'captions_with_splits.csv'
CHECKPOINTS_DIR = PROJECT_DIR / 'checkpoints'
RESULTS_DIR     = PROJECT_DIR / 'results'
CHECKPOINTS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# Local SSD copy avoids Drive I/O bottleneck during training
LOCAL_DATA  = Path('/content/data')
LOCAL_IMAGES      = LOCAL_DATA / 'images'
LOCAL_MASKS_CLASS = LOCAL_DATA / 'masks_class'

if not LOCAL_DATA.exists():
    print('Copying data to local SSD...')
    shutil.copytree(DATA_DIR_DRIVE / 'images', LOCAL_IMAGES)
    shutil.copytree(DATA_DIR_DRIVE / 'masks_class', LOCAL_MASKS_CLASS)
    print('Done.')
else:
    print('Local data already present.')

print(f'Images: {LOCAL_IMAGES}')
print(f'Masks : {LOCAL_MASKS_CLASS}')

Copying data to local SSD...
Done.
Images: /content/data/images
Masks : /content/data/masks_class


In [4]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

Device: cuda


In [5]:
# Class definitions
CLASS_NAMES = ['Tree', 'Shrub', 'Grass', 'Crop', 'Built-up', 'Barren', 'Water']
NUM_CLASSES = len(CLASS_NAMES)

In [6]:
# Load the split CSV from Drive
split_df = pd.read_csv(SPLITS_CSV)
train_df = split_df[split_df['split'] == 'train'].reset_index(drop=True)
val_df   = split_df[split_df['split'] == 'val'].reset_index(drop=True)
test_df  = split_df[split_df['split'] == 'test'].reset_index(drop=True)
print(f'Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}')

Train: 7000  |  Val: 1500  |  Test: 1500


In [7]:
# Class weights via median frequency balancing, computed from the training split.
# Used for weighted cross-entropy loss in all three baselines.
class_avg = train_df[CLASS_NAMES].mean().values
class_freq = class_avg / class_avg.sum()
median_freq = np.median(class_freq)
class_weights = median_freq / (class_freq + 1e-8)

print(f"{'Class':<12} {'Avg %':>8} {'Frequency':>10} {'Weight':>8}")
print('-' * 42)
for i, c in enumerate(CLASS_NAMES):
    print(f'{c:<12} {class_avg[i]:>7.1f}% {class_freq[i]:>10.4f} {class_weights[i]:>8.2f}')

class_weights_tensor = torch.FloatTensor(class_weights).to(device)

Class           Avg %  Frequency   Weight
------------------------------------------
Tree            28.7%     0.2873     0.15
Shrub            0.8%     0.0083     5.12
Grass           45.3%     0.4533     0.09
Crop            18.0%     0.1803     0.24
Built-up         1.1%     0.0113     3.77
Barren           4.3%     0.0426     1.00
Water            1.7%     0.0168     2.54


## 2. Dataset and DataLoaders

The dataset reads pre-converted class-index masks from `data/masks_class/` (produced in `02_preprocessing.ipynb`), so no per-batch RGB conversion is needed.

Augmentations applied jointly to image and mask:
- Random horizontal flip (p=0.5)
- Random vertical flip (p=0.5)
- Random 90° rotation (one of {0°, 90°, 180°, 270°})

Validation and test sets are not augmented. Images are normalized with ImageNet statistics (the backbones expect this).

In [8]:
# Dataset class — reads pre-converted class-index masks directly
class RSDataset(Dataset):
    def __init__(self, dataframe, images_dir, masks_dir, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.masks_dir  = Path(masks_dir)
        self.augment = augment
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )

    def __len__(self):
        return len(self.df)

    def _augment(self, img, mask):
        # img: tensor (3, H, W), mask: tensor (H, W). Apply same transform to both.
        if torch.rand(1).item() < 0.5:
            img  = torch.flip(img,  dims=[2])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            img  = torch.flip(img,  dims=[1])
            mask = torch.flip(mask, dims=[0])
        k = torch.randint(0, 4, (1,)).item()  # 0, 90, 180, 270 degrees
        if k > 0:
            img  = torch.rot90(img,  k, dims=[1, 2])
            mask = torch.rot90(mask, k, dims=[0, 1])
        return img, mask

    def __getitem__(self, idx):
        fname = self.df.iloc[idx]['filename']
        img = Image.open(self.images_dir / fname).convert('RGB')
        img = transforms.functional.to_tensor(img)  # (3, H, W) in [0, 1]
        mask = np.array(Image.open(self.masks_dir / fname))  # (H, W) uint8 with values 0..6
        mask = torch.from_numpy(mask).long()

        if self.augment:
            img, mask = self._augment(img, mask)

        img = self.normalize(img)
        return img, mask

In [11]:
# DataLoader config. Train datasets are recreated per section to toggle augmentation.
BATCH_SIZE  = 16
NUM_WORKERS = 4

# Val and test never get augmented, so we create them once
val_dataset   = RSDataset(val_df,   LOCAL_IMAGES, LOCAL_MASKS_CLASS, augment=False)
test_dataset  = RSDataset(test_df,  LOCAL_IMAGES, LOCAL_MASKS_CLASS, augment=False)

val_loader   = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Val:  {len(val_dataset)} samples, {len(val_loader)} batches')
print(f'Test: {len(test_dataset)} samples, {len(test_loader)} batches')

Val:  1500 samples, 94 batches
Test: 1500 samples, 94 batches


In [12]:
# Confirm image size is divisible by 32 (required by SegFormer's downsampling).
sample_img, sample_mask = val_dataset[0]
H, W = sample_img.shape[-2:]
print(f'Image size: {H} x {W}')
print(f'Mask size : {sample_mask.shape}')
assert H % 32 == 0 and W % 32 == 0, f'Size {H}x{W} is not divisible by 32'
print('Divisible by 32 ✓')

Image size: 256 x 256
Mask size : torch.Size([256, 256])
Divisible by 32 ✓


## 3. UNetFormer architecture

UNetFormer (Wang et al., ISPRS 2022) is a CNN–transformer hybrid for remote sensing semantic segmentation. The encoder is a `swsl_resnet18` backbone (ImageNet-pretrained) and the decoder uses Global-Local Transformer Blocks (GLTB) that combine window self-attention with depthwise convolutions, plus a Feature Refinement Head and an auxiliary head for deep supervision.

Source: https://github.com/WangLibo1995/GeoSeg

In [13]:
# Building blocks (Conv + BN + ReLU variants)
from einops import rearrange
from timm.models.layers import DropPath, trunc_normal_
import timm


class ConvBNReLU(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1,
                 norm_layer=nn.BatchNorm2d, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2),
            norm_layer(out_channels), nn.ReLU6())


class ConvBN(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1,
                 norm_layer=nn.BatchNorm2d, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2),
            norm_layer(out_channels))


class Conv(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2))


class SeparableConvBN(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, dilation=1,
                 norm_layer=nn.BatchNorm2d):
        super().__init__(
            nn.Conv2d(in_channels, in_channels, kernel_size, stride=stride, dilation=dilation,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2,
                      groups=in_channels, bias=False),
            norm_layer(out_channels),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False))

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [14]:
# MLP and Global-Local Attention (window self-attention + local conv branch)
class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.ReLU6, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Conv2d(in_features, hidden_features, 1, 1, 0, bias=True)
        self.act = act_layer()
        self.fc2 = nn.Conv2d(hidden_features, out_features, 1, 1, 0, bias=True)
        self.drop = nn.Dropout(drop, inplace=True)

    def forward(self, x):
        x = self.fc1(x); x = self.act(x); x = self.drop(x)
        x = self.fc2(x); x = self.drop(x); return x


class GlobalLocalAttention(nn.Module):
    def __init__(self, dim=256, num_heads=16, qkv_bias=False,
                 window_size=8, relative_pos_embedding=True):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // self.num_heads
        self.scale = head_dim ** -0.5
        self.ws = window_size
        self.qkv = Conv(dim, 3 * dim, kernel_size=1, bias=qkv_bias)
        self.local1 = ConvBN(dim, dim, kernel_size=3)
        self.local2 = ConvBN(dim, dim, kernel_size=1)
        self.proj = SeparableConvBN(dim, dim, kernel_size=window_size)
        self.attn_x = nn.AvgPool2d(kernel_size=(window_size, 1), stride=1,
                                    padding=(window_size // 2 - 1, 0))
        self.attn_y = nn.AvgPool2d(kernel_size=(1, window_size), stride=1,
                                    padding=(0, window_size // 2 - 1))
        self.relative_pos_embedding = relative_pos_embedding
        if self.relative_pos_embedding:
            self.relative_position_bias_table = nn.Parameter(
                torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads))
            coords_h = torch.arange(self.ws); coords_w = torch.arange(self.ws)
            coords = torch.stack(torch.meshgrid([coords_h, coords_w]))
            coords_flatten = torch.flatten(coords, 1)
            relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
            relative_coords = relative_coords.permute(1, 2, 0).contiguous()
            relative_coords[:, :, 0] += self.ws - 1; relative_coords[:, :, 1] += self.ws - 1
            relative_coords[:, :, 0] *= 2 * self.ws - 1
            relative_position_index = relative_coords.sum(-1)
            self.register_buffer("relative_position_index", relative_position_index)
            trunc_normal_(self.relative_position_bias_table, std=.02)

    def pad(self, x, ps):
        _, _, H, W = x.size()
        if W % ps != 0: x = F.pad(x, (0, ps - W % ps), mode='reflect')
        if H % ps != 0: x = F.pad(x, (0, 0, 0, ps - H % ps), mode='reflect')
        return x

    def pad_out(self, x):
        return F.pad(x, pad=(0, 1, 0, 1), mode='reflect')

    def forward(self, x):
        B, C, H, W = x.shape
        local = self.local2(x) + self.local1(x)
        x = self.pad(x, self.ws); B, C, Hp, Wp = x.shape
        qkv = self.qkv(x)
        q, k, v = rearrange(qkv, 'b (qkv h d) (hh ws1) (ww ws2) -> qkv (b hh ww) h (ws1 ws2) d',
            h=self.num_heads, d=C // self.num_heads, hh=Hp // self.ws, ww=Wp // self.ws,
            qkv=3, ws1=self.ws, ws2=self.ws)
        dots = (q @ k.transpose(-2, -1)) * self.scale
        if self.relative_pos_embedding:
            relative_position_bias = self.relative_position_bias_table[
                self.relative_position_index.view(-1)].view(self.ws * self.ws, self.ws * self.ws, -1)
            relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
            dots += relative_position_bias.unsqueeze(0)
        attn = dots.softmax(dim=-1); attn = attn @ v
        attn = rearrange(attn, '(b hh ww) h (ws1 ws2) d -> b (h d) (hh ws1) (ww ws2)',
            h=self.num_heads, d=C // self.num_heads, hh=Hp // self.ws, ww=Wp // self.ws,
            ws1=self.ws, ws2=self.ws)
        attn = attn[:, :, :H, :W]
        out = self.attn_x(F.pad(attn, pad=(0, 0, 0, 1), mode='reflect')) + \
              self.attn_y(F.pad(attn, pad=(0, 1, 0, 0), mode='reflect'))
        out = out + local; out = self.pad_out(out); out = self.proj(out)
        out = out[:, :, :H, :W]
        return out

In [15]:
# Transformer Block (GLTB): pre-norm, attention, residual, MLP, residual
class Block(nn.Module):
    def __init__(self, dim=256, num_heads=16, mlp_ratio=4., qkv_bias=False,
                 drop=0., drop_path=0.,
                 act_layer=nn.ReLU6, norm_layer=nn.BatchNorm2d, window_size=8):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = GlobalLocalAttention(dim, num_heads=num_heads, qkv_bias=qkv_bias,
                                         window_size=window_size)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, out_features=dim,
                       act_layer=act_layer, drop=drop)
        self.norm2 = norm_layer(dim)

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

In [16]:
# Decoder components: weighted feature fusion (WF), feature refinement head, aux head
class WF(nn.Module):
    def __init__(self, in_channels=128, decode_channels=128, eps=1e-8):
        super().__init__()
        self.pre_conv = Conv(in_channels, decode_channels, kernel_size=1)
        self.weights = nn.Parameter(torch.ones(2, dtype=torch.float32), requires_grad=True)
        self.eps = eps
        self.post_conv = ConvBNReLU(decode_channels, decode_channels, kernel_size=3)

    def forward(self, x, res):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        weights = nn.ReLU()(self.weights)
        fuse_weights = weights / (torch.sum(weights, dim=0) + self.eps)
        x = fuse_weights[0] * self.pre_conv(res) + fuse_weights[1] * x
        return self.post_conv(x)


class FeatureRefinementHead(nn.Module):
    def __init__(self, in_channels=64, decode_channels=64):
        super().__init__()
        self.pre_conv = Conv(in_channels, decode_channels, kernel_size=1)
        self.weights = nn.Parameter(torch.ones(2, dtype=torch.float32), requires_grad=True)
        self.eps = 1e-8
        self.post_conv = ConvBNReLU(decode_channels, decode_channels, kernel_size=3)
        self.pa = nn.Sequential(
            nn.Conv2d(decode_channels, decode_channels, kernel_size=3, padding=1,
                      groups=decode_channels), nn.Sigmoid())
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            Conv(decode_channels, decode_channels // 16, kernel_size=1), nn.ReLU6(),
            Conv(decode_channels // 16, decode_channels, kernel_size=1), nn.Sigmoid())
        self.shortcut = ConvBN(decode_channels, decode_channels, kernel_size=1)
        self.proj = SeparableConvBN(decode_channels, decode_channels, kernel_size=3)
        self.act = nn.ReLU6()

    def forward(self, x, res):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        weights = nn.ReLU()(self.weights)
        fuse_weights = weights / (torch.sum(weights, dim=0) + self.eps)
        x = fuse_weights[0] * self.pre_conv(res) + fuse_weights[1] * x
        x = self.post_conv(x); shortcut = self.shortcut(x)
        pa = self.pa(x) * x; ca = self.ca(x) * x
        x = pa + ca; x = self.proj(x) + shortcut
        return self.act(x)


class AuxHead(nn.Module):
    def __init__(self, in_channels=64, num_classes=8):
        super().__init__()
        self.conv = ConvBNReLU(in_channels, in_channels)
        self.drop = nn.Dropout(0.1)
        self.conv_out = Conv(in_channels, num_classes, kernel_size=1)

    def forward(self, x, h, w):
        feat = self.conv(x); feat = self.drop(feat); feat = self.conv_out(feat)
        return F.interpolate(feat, size=(h, w), mode='bilinear', align_corners=False)

In [18]:
# Full Decoder + UNetFormer
class Decoder(nn.Module):
    def __init__(self, encoder_channels=(64, 128, 256, 512), decode_channels=64,
                 dropout=0.1, window_size=8, num_classes=7):
        super().__init__()
        self.pre_conv = ConvBN(encoder_channels[-1], decode_channels, kernel_size=1)
        self.b4 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p3 = WF(encoder_channels[-2], decode_channels)
        self.b3 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p2 = WF(encoder_channels[-3], decode_channels)
        self.b2 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p1 = FeatureRefinementHead(encoder_channels[-4], decode_channels)
        self.up4 = nn.UpsamplingBilinear2d(scale_factor=4)
        self.up3 = nn.UpsamplingBilinear2d(scale_factor=2)
        self.aux_head = AuxHead(decode_channels, num_classes)
        self.segmentation_head = nn.Sequential(
            ConvBNReLU(decode_channels, decode_channels),
            nn.Dropout2d(p=dropout, inplace=True),
            Conv(decode_channels, num_classes, kernel_size=1))
        self.init_weight()

    def forward(self, res1, res2, res3, res4, h, w):
        if self.training:
            x = self.b4(self.pre_conv(res4)); h4 = self.up4(x)
            x = self.p3(x, res3); x = self.b3(x); h3 = self.up3(x)
            x = self.p2(x, res2); x = self.b2(x); h2 = x
            x = self.p1(x, res1); x = self.segmentation_head(x)
            x = F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)
            ah = h4 + h3 + h2; ah = self.aux_head(ah, h, w)
            return x, ah
        else:
            x = self.b4(self.pre_conv(res4))
            x = self.p3(x, res3); x = self.b3(x)
            x = self.p2(x, res2); x = self.b2(x)
            x = self.p1(x, res1); x = self.segmentation_head(x)
            return F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)

    def init_weight(self):
        for m in self.children():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, a=1)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)


class UNetFormer(nn.Module):
    def __init__(self, decode_channels=64, dropout=0.1, backbone_name='swsl_resnet18',
                 pretrained=True, window_size=8, num_classes=7):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, features_only=True, output_stride=32,
                                          out_indices=(1, 2, 3, 4), pretrained=pretrained)
        encoder_channels = self.backbone.feature_info.channels()
        self.decoder = Decoder(encoder_channels, decode_channels, dropout, window_size, num_classes)

    def forward(self, x):
        h, w = x.size()[-2:]
        res1, res2, res3, res4 = self.backbone(x)
        if self.training:
            x, ah = self.decoder(res1, res2, res3, res4, h, w)
            return x, ah
        else:
            return self.decoder(res1, res2, res3, res4, h, w)

In [19]:
# Quick verification: instantiate, count parameters, forward pass in both modes
test_model = UNetFormer(num_classes=NUM_CLASSES).to(device)
total_params     = sum(p.numel() for p in test_model.parameters())
trainable_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f'Total parameters    : {total_params / 1e6:.2f}M')
print(f'Trainable parameters: {trainable_params / 1e6:.2f}M')

with torch.no_grad():
    sample_batch = next(iter(val_loader))[0][:2].to(device)

    test_model.eval()
    out_eval = test_model(sample_batch)
    print(f'Eval  mode: input {sample_batch.shape} -> output {out_eval.shape}')

    test_model.train()
    out_main, out_aux = test_model(sample_batch)
    print(f'Train mode: main {out_main.shape}, aux {out_aux.shape}')

del test_model  # free memory before actual training
torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name swsl_resnet18 to current resnet18.fb_swsl_ig1b_ft_in1k.
  model = create_fn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Total parameters    : 11.73M
Trainable parameters: 11.73M


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Eval  mode: input torch.Size([2, 3, 256, 256]) -> output torch.Size([2, 7, 256, 256])
Train mode: main torch.Size([2, 7, 256, 256]), aux torch.Size([2, 7, 256, 256])


## 4. UNetFormer baseline (no augmentation)

Reproduces the PoC baseline. Same hyperparameters, same architecture, no augmentation, but using pre-converted class-index masks from `02_preprocessing`. Target: validation mIoU ≈ 0.36 (PoC value).

**Training setup:**
- 30 epochs, AdamW (lr=6e-4, weight_decay=0.01)
- CosineAnnealingLR schedule (eta_min=1e-6)
- Weighted CE (median frequency balancing) + 0.4 × auxiliary head loss

In [20]:
# Training hyperparameters (shared across all baselines)
NUM_EPOCHS      = 30
LR              = 6e-4
WEIGHT_DECAY    = 0.01
AUX_LOSS_WEIGHT = 0.4

In [21]:
# Metrics: per-class IoU, mean IoU, overall accuracy
def compute_metrics(pred, target, num_classes=NUM_CLASSES):
    """Per-image metrics. pred and target are tensors of shape (H, W) with class indices."""
    ious = []
    for cls in range(num_classes):
        pred_cls   = (pred == cls)
        target_cls = (target == cls)
        intersection = (pred_cls & target_cls).sum().item()
        union        = (pred_cls | target_cls).sum().item()
        ious.append(float('nan') if union == 0 else intersection / union)
    valid_ious = [x for x in ious if not np.isnan(x)]
    miou = np.mean(valid_ious) if valid_ious else 0.0
    oa = (pred == target).sum().item() / target.numel()
    return ious, miou, oa

In [22]:
# Training and evaluation loops for UNetFormer (uses aux head during training)
def train_one_epoch_unet(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits_main, logits_aux = model(imgs)
        loss = criterion(logits_main, masks) + AUX_LOSS_WEIGHT * criterion(logits_aux, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion, device, num_classes=NUM_CLASSES):
    """Returns: per-class IoU list, mIoU, overall accuracy, average loss."""
    model.eval()
    all_ious = [[] for _ in range(num_classes)]
    total_correct = 0
    total_pixels  = 0
    total_loss    = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs)
        total_loss += criterion(logits, masks).item()
        preds = logits.argmax(dim=1).cpu()
        masks_cpu = masks.cpu()
        for pred, mask in zip(preds, masks_cpu):
            ious, _, _ = compute_metrics(pred, mask, num_classes)
            for c in range(num_classes):
                if not np.isnan(ious[c]):
                    all_ious[c].append(ious[c])
            total_correct += (pred == mask).sum().item()
            total_pixels  += mask.numel()
    class_ious = [np.mean(all_ious[c]) if all_ious[c] else float('nan') for c in range(num_classes)]
    valid = [x for x in class_ious if not np.isnan(x)]
    miou = np.mean(valid) if valid else 0.0
    oa = total_correct / total_pixels
    return class_ious, miou, oa, total_loss / len(loader)

In [23]:
# Reusable training function — called for each baseline run
def train_baseline(model, train_loader, val_loader, num_epochs, lr, save_path, run_name):
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    history = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'val_oa': []}
    best_miou = 0.0

    print(f'Training {run_name} for {num_epochs} epochs...')
    print(f"{'Epoch':>5} {'T_Loss':>8} {'V_Loss':>8} {'V_mIoU':>8} {'V_OA':>8} {'Time':>7} {'':>6}")
    print('-' * 55)

    for epoch in range(num_epochs):
        t0 = time.time()
        train_loss = train_one_epoch_unet(model, train_loader, optimizer, criterion, device)
        _, val_miou, val_oa, val_loss = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_miou'].append(val_miou)
        history['val_oa'].append(val_oa)

        status = ''
        if val_miou > best_miou:
            best_miou = val_miou
            torch.save(model.state_dict(), save_path)
            status = 'BEST'

        elapsed = time.time() - t0
        print(f'{epoch+1:>5} {train_loss:>8.4f} {val_loss:>8.4f} {val_miou:>8.4f} {val_oa:>8.4f} {elapsed:>6.1f}s {status:>6}')

    print(f'\nBest validation mIoU: {best_miou:.4f}')
    return history, best_miou

In [24]:
# Train UNetFormer without augmentation (PoC reproduction)
train_dataset_no_aug = RSDataset(train_df, LOCAL_IMAGES, LOCAL_MASKS_CLASS, augment=False)
train_loader_no_aug  = DataLoader(train_dataset_no_aug, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=True)

model_unet_noaug = UNetFormer(num_classes=NUM_CLASSES).to(device)

history_unet_noaug, best_miou_unet_noaug = train_baseline(
    model_unet_noaug, train_loader_no_aug, val_loader,
    num_epochs=NUM_EPOCHS, lr=LR,
    save_path=str(CHECKPOINTS_DIR / 'unetformer_noaug_best.pth'),
    run_name='UNetFormer (no aug)',
)

Training UNetFormer (no aug) for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.2275   1.0297   0.1575   0.5584   25.2s   BEST
    2   0.9117   0.5100   0.2212   0.6618   24.0s   BEST
    3   0.8049   0.6605   0.1975   0.5947   23.9s       
    4   0.7133   0.4186   0.2897   0.7761   23.5s   BEST
    5   0.7042   0.4668   0.2400   0.7154   23.9s       
    6   0.6678   0.3722   0.2837   0.7605   23.7s       
    7   0.6099   0.4911   0.2676   0.7185   23.8s       
    8   0.6031   0.4583   0.2586   0.7536   23.7s       
    9   0.5662   0.5228   0.2575   0.7522   24.2s       
   10   0.5213   0.3328   0.2891   0.7765   23.6s       
   11   0.5099   0.4045   0.3090   0.8081   23.7s   BEST
   12   0.4734   0.3206   0.3031   0.8075   23.5s       
   13   0.4682   0.3489   0.2966   0.7762   24.1s       
   14   0.4567   0.3019   0.3305   0.8345   23.7s   BEST
   15   0.4178   0.3892   0.2873   0.7683  

In [25]:
# Persist history to disk so we don't lose it if Colab disconnects
import json

def save_history(history, name):
    path = RESULTS_DIR / f'{name}_history.json'
    with open(path, 'w') as f:
        json.dump(history, f)
    print(f'Saved history to {path}')

save_history(history_unet_noaug, 'unetformer_noaug')

Saved history to /content/drive/MyDrive/DI725_Project/results/unetformer_noaug_history.json
